In [3]:
import pandas as pd

# Loading the file
df_comed = pd.read_csv('COMED_hourly.csv', parse_dates=['Datetime'])

# Checking the timeframe
print(df_comed['Datetime'].min(), '→', df_comed['Datetime'].max())

# Making sure that the data is sorted
df_comed = df_comed.sort_values('Datetime')

# Setting time index
df_comed = df_comed.set_index('Datetime')

# Checking for regularity and missign intervals
print("\nResampled to hourly, missing intervals:")
print(df_comed.resample('h').size().value_counts())

2011-01-01 01:00:00 → 2018-08-03 00:00:00

Resampled to hourly, missing intervals:
1    66489
0       11
2        4
Name: count, dtype: int64


In [4]:
# Finding all hours with more than 1 record
duplicated_hours = df_comed.index[df_comed.index.duplicated(keep=False)]

# Printing 
print(df_comed.loc[duplicated_hours])

                     COMED_MW
Datetime                     
2014-11-02 02:00:00    9184.0
2014-11-02 02:00:00    8869.0
2014-11-02 02:00:00    9184.0
2014-11-02 02:00:00    8869.0
2015-11-01 02:00:00    8270.0
2015-11-01 02:00:00    7923.0
2015-11-01 02:00:00    8270.0
2015-11-01 02:00:00    7923.0
2016-11-06 02:00:00    8028.0
2016-11-06 02:00:00    7814.0
2016-11-06 02:00:00    8028.0
2016-11-06 02:00:00    7814.0
2017-11-05 02:00:00    8198.0
2017-11-05 02:00:00    7878.0
2017-11-05 02:00:00    8198.0
2017-11-05 02:00:00    7878.0


The problem of dublicated accures because of the winter and summer time shifts. So some hours appear several time, since they are consequtive, we will take the mean of them

In [5]:
df_comed = df_comed.groupby(df_comed.index).mean()

In [6]:
duplicated_hours = df_comed.index[df_comed.index.duplicated(keep=False)]
print(df_comed.loc[duplicated_hours])

Empty DataFrame
Columns: [COMED_MW]
Index: []


In [7]:
print("\nResampled to hourly, missing intervals:")
print(df_comed.resample('h').size().value_counts())


Resampled to hourly, missing intervals:
1    66493
0       11
Name: count, dtype: int64


In [8]:
# Finding the missing hours
full_range = pd.date_range(
    start=df_comed.index.min(),
    end=df_comed.index.max(),
    freq='h'
)

missing_hours = full_range.difference(df_comed.index)
print(f"Missing hours ({len(missing_hours)}):")
print(missing_hours)

Missing hours (11):
DatetimeIndex(['2011-03-13 03:00:00', '2011-11-06 02:00:00',
               '2012-03-11 03:00:00', '2012-11-04 02:00:00',
               '2013-03-10 03:00:00', '2013-11-03 02:00:00',
               '2014-03-09 03:00:00', '2015-03-08 03:00:00',
               '2016-03-13 03:00:00', '2017-03-12 03:00:00',
               '2018-03-11 03:00:00'],
              dtype='datetime64[ns]', freq=None)


This hours are missed because they simply do not exist due to the time shiftts

## Loading weather dataset